# LAK-10 (Deep) — Iceberg internals

Everything earlier in this track *used* Iceberg's metadata; this notebook **opens it up**. There is
nothing to break here — each step is a short demo of a real internal, and **the metadata-table (or
on-disk) output _is_ the artifact**. We walk the metadata tree:

```
catalog pointer ─► vN.metadata.json ─► manifest list ─► manifest files ─► data + delete files
```

**Pre-requisite:** the unified Spark server is up (`make up`); this connects via Spark Connect.
**Connect-safe:** `spark.sql` + DataFrame + read-only `os`/`glob` only (no `sparkContext`/RDD).
**Laptop-safe:** two tiny tables under `.tmp/local_iceberg_warehouse/default/`; dropped at the end
(`make clean` clears the rest). Companion: [`README.md`](./README.md) and
[`docs/CURRICULUM_PLAN.md`](../../docs/CURRICULUM_PLAN.md) (LAK-10 + the Iceberg "Niche/Deep" inventory).

In [ ]:
from common.spark_session import spark, display_df
from pyspark.sql import functions as F
import os, glob
import common

# Hadoop (filesystem) catalog — see conf/spark-defaults.conf. The warehouse is just a directory,
# which is exactly why we can read the on-disk metadata tree below.
# Anchor to the repo root: a notebook kernel's CWD is its own dir, not the repo root,
# so a relative ".tmp/..." would miss the server's warehouse. Derive root from `common`.
_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(common.__file__)))
WAREHOUSE = os.path.join(_ROOT, ".tmp/local_iceberg_warehouse")
T  = "iceberg_catalog.default.lak10_orders"      # v2 table we evolve through the demos
T1 = "iceberg_catalog.default.lak10_orders_v1"   # v1 twin (for the v1-vs-v2 contrast in demo 3)
TDIR = os.path.join(WAREHOUSE, "default", "lak10_orders")   # on-disk path for os/glob
spark

## Step 0 — a small partitioned v2 table, written twice

A tiny `orders` table partitioned by `order_date`, **format-version 2** (so demo 3 can show
merge-on-read delete files), then a second append. Two commits = two snapshots = a metadata pointer
that has already moved.

In [ ]:
for t in (T, T1):
    spark.sql(f"DROP TABLE IF EXISTS {t}")

# v2, merge-on-read, partitioned by order_date
spark.sql(f"""
    CREATE TABLE {T} (
        order_id     BIGINT,
        customer_id  BIGINT,
        order_date   DATE,
        amount       DOUBLE
    ) USING iceberg
    PARTITIONED BY (order_date)
    TBLPROPERTIES (
        'format-version' = '2',
        'write.delete.mode' = 'merge-on-read',
        'write.update.mode' = 'merge-on-read',
        'write.merge.mode'  = 'merge-on-read'
    )
""")

def batch(start, end, day):
    return (spark.range(start, end).withColumnRenamed("id", "order_id")
            .withColumn("customer_id", F.pmod(F.col("order_id"), F.lit(20)))
            .withColumn("order_date", F.lit(day).cast("date"))
            .withColumn("amount", F.round(F.rand(seed=1) * 100, 2)))

batch(1, 51,  "2026-01-01").writeTo(T).append()   # snapshot 1
batch(51, 101, "2026-01-02").writeTo(T).append()  # snapshot 2
print("rows:", spark.table(T).count(), "across",
      spark.sql(f"SELECT COUNT(*) AS n FROM {T}.partitions").first()["n"], "partitions")

## 1. Metadata pointer / version-hint

The whole table is reached through **one tiny pointer**. In the Hadoop catalog that pointer is the
file `metadata/version-hint.text`, holding a single integer **N** → "current state is
`vN.metadata.json`". Every commit writes a *new* `vN.metadata.json` (the schema, partition specs,
snapshot list, and refs at that moment) and bumps the hint. We list the `metadata/` dir, read the
hint, then show Iceberg's own view of the same chain (`<t>.metadata_log_entries`).

In [ ]:
meta_dir = os.path.join(TDIR, "metadata")
print("Files in", meta_dir, ":")
for name in sorted(os.listdir(meta_dir)):
    print("   ", name)

current_n = open(os.path.join(meta_dir, "version-hint.text")).read().strip()
versions = sorted(p.split("/")[-1] for p in glob.glob(os.path.join(meta_dir, "v*.metadata.json")))
print(f"\nversion-hint.text -> {current_n}   (current = v{current_n}.metadata.json)")
print("metadata.json chain:", versions, f"  ({len(versions)} commits incl. CREATE)")

print("\n<t>.metadata_log_entries (one row per vN.metadata.json ever written):")
display_df(spark.sql(f"SELECT timestamp, file FROM {T}.metadata_log_entries ORDER BY timestamp"))

**What you just saw.** A `vN.metadata.json` per commit (CREATE + two appends), a single
`version-hint.text` naming the current one, and the manifest-list (`snap-*.avro`) / manifest
(`*.avro`) files those point at. `.metadata_log_entries` is the same pointer history as a SQL table.
Reading the table = read the hint → open that one JSON → follow it down.

## 2. Manifest column stats & pruning

Each data file is tracked in a manifest **with per-column statistics** — min/max bounds, null/value
counts. Iceberg uses min/max to **skip whole files** for a `WHERE` *before opening them*. In
`<t>.files`, `readable_metrics` is the human-readable view; `lower_bounds`/`upper_bounds` are the
raw per-column bounds.

In [ ]:
display_df(spark.sql(f"""
    SELECT file_path, record_count, readable_metrics
    FROM {T}.files
"""))

**What you just saw.** Every data file carries each column's `{min, max, null_count,
value_count}`. The table is partitioned by `order_date`, so each file's `order_date` min == max ==
its partition — a `WHERE order_date = DATE '2026-01-02'` compares that literal to each file's bounds
and **reads only the matching file(s)**. The same min/max trick prunes *non-partition* columns too
(e.g. `amount`), so Iceberg skips files even without partitioning — never opening a Parquet footer
for files it can prove can't match.

## 3. Format v1 vs v2 — delete files (merge-on-read)

A row-level `DELETE`/`UPDATE`/`MERGE` can be applied two ways. **Copy-on-write** rewrites the whole
data file without the row (the LAK-8 cost). **Merge-on-read** (our v2 table) instead writes a small
**delete file** — "ignore these rows" — and reconciles at read time: cheap to write, slightly
costlier to read. In `<t>.files`, `content` distinguishes them: **0 = data, 1 = position delete,
2 = equality delete**. Format **v1 cannot represent delete files at all**.

In [ ]:
# Delete one row from the v2 table -> merge-on-read writes a DELETE FILE, not a rewrite
spark.sql(f"DELETE FROM {T} WHERE order_id = 1")

display_df(spark.sql(f"""
    SELECT content,                       -- 0=data, 1=position delete, 2=equality delete
           CASE content WHEN 0 THEN 'data'
                        WHEN 1 THEN 'position-delete'
                        WHEN 2 THEN 'equality-delete' END AS content_kind,
           COUNT(*)            AS files,
           SUM(record_count)   AS records
    FROM {T}.files
    GROUP BY content
    ORDER BY content
"""))

In [ ]:
# v1 twin: no merge-on-read, so a delete is copy-on-write and there are NO delete files
spark.sql(f"""
    CREATE TABLE {T1} (order_id BIGINT, amount DOUBLE) USING iceberg
    TBLPROPERTIES ('format-version' = '1')
""")
spark.range(1, 11).selectExpr("id AS order_id", "id * 1.0 AS amount").writeTo(T1).append()
spark.sql(f"DELETE FROM {T1} WHERE order_id = 1")

v1_del = spark.sql(f"SELECT COUNT(*) AS n FROM {T1}.files WHERE content != 0").first()["n"]
v2_del = spark.sql(f"SELECT COUNT(*) AS n FROM {T}.files  WHERE content != 0").first()["n"]
print(f"v1 delete files: {v1_del}   (copy-on-write: the data file was rewritten)")
print(f"v2 delete files: {v2_del}   (merge-on-read: a delete file was written)")

**What you just saw.** On v2, the delete added a **`position-delete`** file (`content = 1`)
instead of rewriting data; the row is filtered out at read time. On the **v1** twin the identical
delete produced **zero** delete files — it had to rewrite the data file (copy-on-write). That choice
is the heart of the CoW-vs-MoR tradeoff (LAK-8).

## 4. Snapshots, refs & history

Table history is a **DAG of snapshots** (each commit = one snapshot linked to its parent). **Refs**
are *named pointers* into that DAG: every table has a `main` branch, and you can add **branches**
(isolated write lines) and **tags** (immutable labels on a snapshot — a release marker you can
time-travel to). This is the machinery behind rollback and write-audit-publish.

In [ ]:
print("History (snapshot DAG — parent_id links commits):")
display_df(spark.sql(f"""
    SELECT made_current_at, snapshot_id, parent_id, is_current_ancestor
    FROM {T}.history ORDER BY made_current_at
"""))

# Tag the current snapshot, then list refs: the built-in `main` branch + our tag.
spark.sql(f"ALTER TABLE {T} CREATE TAG `release_2026_01` RETAIN 7 DAYS")
print("\nRefs (named pointers into the snapshot DAG):")
display_df(spark.sql(f"""
    SELECT name, type, snapshot_id, max_reference_age_in_ms
    FROM {T}.refs ORDER BY type, name
"""))

**What you just saw.** `.history` is the snapshot chain (each `parent_id` points at the prior
commit; `is_current_ancestor` marks the live lineage). `.refs` lists the always-present **`main`
BRANCH** plus the **`release_2026_01` TAG** we just created — you could now
`... FOR VERSION AS OF 'release_2026_01'` or roll back to it, and the tag pins that snapshot so
expiry won't drop it. Catalogs like Nessie take branches/tags much further (demo 5).

## 5. Catalog types — Hadoop vs Hive vs REST/Nessie

The **catalog** turns "a tree of files" into "a named table" and makes the **commit** (swapping the
metadata pointer) safe. This repo uses the **Hadoop (filesystem)** catalog — great for *seeing* the
tree, but its commit is a filesystem rename, so it can't safely coordinate concurrent writers on
object storage. The cell prints the live config.

In [ ]:
# The actual catalog config driving this notebook (from conf/spark-defaults.conf):
print("catalog impl =", spark.conf.get("spark.sql.catalog.iceberg_catalog", "(unset)"))
for k in ("type", "warehouse"):
    print(f"spark.sql.catalog.iceberg_catalog.{k:9} = "
          f"{spark.conf.get(f'spark.sql.catalog.iceberg_catalog.{k}', '(unset)')}")

**What you just saw.** `type = hadoop`, `warehouse = ./.tmp/local_iceberg_warehouse` — a plain
directory catalog. How the common catalogs compare:

| Catalog | Pointer / commit mechanism | Atomic multi-writer? | Multi-engine? | Branching |
|---------|----------------------------|----------------------|---------------|-----------|
| **Hadoop** (this repo) | `version-hint.text` + filesystem rename | Only on a filesystem with atomic rename (not most object stores) | Anyone with FS access | Refs exist, no shared coordination |
| **Hive** (HMS) | Pointer row in the Hive Metastore, updated in a metastore txn | Yes (serialized by HMS) | Yes (Spark/Trino/Flink share HMS) | Refs only |
| **REST** (Iceberg REST spec) | A REST service owns the pointer; commit = atomic API call | Yes (service serializes commits) | Yes (engine-neutral protocol) | Refs; backend-dependent |
| **Nessie** | Git-like versioned catalog; commit on a branch | Yes | Yes | **First-class branches/tags/merge across many tables** |

The Hadoop catalog is ideal for *learning the on-disk format* (it's why we can read
`version-hint.text` and the `vN.metadata.json` chain); production reaches for Hive/REST/Nessie to get
**atomic commits, safe multi-engine concurrency, and real branching**.

## 6. Manifest list & partition summaries

Between the snapshot and the data files sits the **manifest list** (one `snap-*.avro` per snapshot,
listing that snapshot's manifests). Each entry carries **`partition_summaries`** — the range of
partition values that manifest covers. So planning a partition filter can **drop an entire
manifest** (and everything it lists) before reading any file entry. `<t>.manifests` exposes it.

In [ ]:
display_df(spark.sql(f"""
    SELECT path,
           added_data_files_count,
           existing_data_files_count,
           deleted_data_files_count,
           partition_summaries
    FROM {T}.manifests
"""))

**What you just saw.** Each manifest row gives its data-file counts and a `partition_summaries`
array with the `lower_bound`/`upper_bound` of `order_date`. That's the **outer** layer of pruning:
a `WHERE order_date = '2026-01-02'` can skip any manifest whose summary excludes that date, *then*
(demo 2) skip individual files by their column bounds. Manifest list → manifest summary → file
bounds is a three-level funnel, and this is the top of it.

> **At scale:** when a table has thousands of manifests (the LAK-5 "manifest explosion" pathology),
> these per-manifest summaries keep planning cheap — and `rewrite_manifests` reorganizes them so the
> summaries stay tight and prunable.

## Takeaways

- **The read path is a funnel of pointers, each pruning:** catalog → `vN.metadata.json` (demo 1) →
  manifest list + `partition_summaries` (demo 6) → manifest + per-file `lower/upper_bounds`
  (demo 2) → data files, with **delete files** layered on for v2 merge-on-read (demo 3).
- **Everything is observable over Spark Connect** via metadata tables: `.metadata_log_entries`,
  `.files` (`readable_metrics`, `content`), `.snapshots`/`.history`/`.refs`, `.manifests`,
  `.partitions` — plus the raw files via `os`/`glob` because we're on the Hadoop catalog.
- **Observable here vs. only in a real catalog:** the *file tree* (version-hint, metadata chain,
  manifest lists) is fully visible on Hadoop; **atomic multi-writer commits, multi-engine
  concurrency, and Git-like branching** need Hive/REST/Nessie (demo 5).
- **Pointers:** [`docs/CURRICULUM_PLAN.md`](../../docs/CURRICULUM_PLAN.md) (LAK-10 + Iceberg
  Niche/Deep inventory); the helper that *measures* these same tables in LAK-2/3/5 is
  [`common/iceberg_meta.py`](../../common/iceberg_meta.py).

## Teardown

We created two real (tiny) tables this time, so drop them. `make clean` clears all of `.tmp/`.

In [ ]:
for t in (T, T1):
    spark.sql(f"DROP TABLE IF EXISTS {t}")
print("Dropped", T, "and", T1, "— (`make clean` clears all of .tmp/).")